# MXFrame Core

> PyArrow-backed DataFrame with MAX Engine execution support.

This module provides:
- `MXFrame`: Core DataFrame class backed by PyArrow Table
- `DTypeMapping`: Arrow ↔ MAX dtype conversion
- Schema introspection and column access

In [2]:
#| default_exp core

In [3]:
#| hide
from nbdev.showdoc import *

In [4]:
#| export
import pyarrow as pa
import numpy as np
from typing import Union, List, Dict, Optional
from dataclasses import dataclass

In [10]:
#| export
# Arrow ↔ MAX dtype mapping
ARROW_TO_MAX_DTYPE = {
    pa.int8(): 'int8',
    pa.int16(): 'int16',
    pa.int32(): 'int32',
    pa.int64(): 'int64',
    pa.uint8(): 'uint8',
    pa.uint16(): 'uint16',
    pa.uint32(): 'uint32',
    pa.uint64(): 'uint64',
    pa.float16(): 'float16',
    pa.float32(): 'float32',
    pa.float64(): 'float64',
    pa.bool_(): 'bool',
    pa.string(): 'string',
    pa.date32(): 'int32',  # Store as epoch days
}

def arrow_to_max_dtype(arrow_type: pa.DataType) -> str:
    """Convert Arrow dtype to MAX dtype string."""
    if arrow_type in ARROW_TO_MAX_DTYPE:
        return ARROW_TO_MAX_DTYPE[arrow_type]
    # Handle parameterized types
    if pa.types.is_date32(arrow_type):
        return 'int32'
    if pa.types.is_date64(arrow_type):
        return 'int64'
    if pa.types.is_timestamp(arrow_type):
        return 'int64'
    raise ValueError(f"Unsupported Arrow type: {arrow_type}")

In [11]:
#| export
@dataclass
class ColumnInfo:
    """Information about a single column."""
    name: str
    dtype: str  # MAX dtype
    arrow_type: pa.DataType
    nullable: bool = True
    
    def __repr__(self):
        null_str = "?" if self.nullable else ""
        return f"{self.name}: {self.dtype}{null_str}"

In [ ]:
#| export
class MXFrame:
    """
    PyArrow-backed DataFrame with MAX Engine execution support.
    
    Example:
        >>> table = pa.table({'qty': [1, 2, 3], 'price': [10.0, 20.0, 30.0]})
        >>> frame = MXFrame(table)
        >>> print(frame.schema)
        >>> print(frame.num_rows)
        >>> print(frame['qty'])
    """
    
    def __init__(self, data: Union[pa.Table, Dict[str, list], 'pd.DataFrame', None] = None):
        """
        Create an MXFrame from various sources.
        
        Args:
            data: PyArrow Table, dict of columns, or Pandas DataFrame
        """
        if data is None:
            self._table = pa.table({})
        elif isinstance(data, pa.Table):
            self._table = data
        elif isinstance(data, dict):
            self._table = pa.table(data)
        else:
            # Try pandas
            try:
                self._table = pa.Table.from_pandas(data)
            except Exception as e:
                raise TypeError(f"Cannot create MXFrame from {type(data)}: {e}")
    
    # ==================== Properties ====================
    
    @property
    def num_rows(self) -> int:
        """Number of rows in the frame."""
        return self._table.num_rows
    
    @property
    def num_columns(self) -> int:
        """Number of columns in the frame."""
        return self._table.num_columns
    
    @property
    def column_names(self) -> List[str]:
        """List of column names."""
        return self._table.column_names
    
    @property
    def shape(self) -> tuple:
        """Shape as (num_rows, num_columns)."""
        return (self.num_rows, self.num_columns)
    
    @property
    def schema(self) -> List[ColumnInfo]:
        """Schema as list of ColumnInfo."""
        result = []
        for field in self._table.schema:
            result.append(ColumnInfo(
                name=field.name,
                dtype=arrow_to_max_dtype(field.type),
                arrow_type=field.type,
                nullable=field.nullable,
            ))
        return result
    
    # ==================== Column Access ====================
    
    def __getitem__(self, key: Union[str, int, List[str]]) -> Union[pa.ChunkedArray, 'MXFrame']:
        """
        Access column(s) by name or index.
        
        Args:
            key: Column name (str), index (int), or list of column names
            
        Returns:
            Single column as ChunkedArray, or new MXFrame with selected columns
        """
        if isinstance(key, str):
            return self._table.column(key)
        elif isinstance(key, int):
            return self._table.column(key)
        elif isinstance(key, list):
            return MXFrame(self._table.select(key))
        else:
            raise TypeError(f"Invalid key type: {type(key)}")
    
    def column(self, name: str) -> pa.ChunkedArray:
        """Get column by name."""
        return self._table.column(name)
    
    def column_to_numpy(self, name: str) -> np.ndarray:
        """Get column as NumPy array (alias for to_numpy for consistency)."""
        return self.to_numpy(name)
    
    # ==================== Conversions ====================
    
    def to_arrow(self) -> pa.Table:
        """Return the underlying PyArrow Table."""
        return self._table
    
    def to_pandas(self):
        """Convert to Pandas DataFrame."""
        return self._table.to_pandas()
    
    def to_numpy(self, column: str) -> np.ndarray:
        """Get column as NumPy array (zero-copy when possible)."""
        return self._table.column(column).to_numpy(zero_copy_only=False)
    
    # ==================== GroupBy ====================
    
    def group_by(self, *keys: str, device: str = "auto"):
        """Group frame by one or more columns.
        
        Args:
            *keys: Column names to group by
            device: Device for compute ('cpu', 'gpu', 'auto')
            
        Returns:
            MXGroupBy object with .agg() method
            
        Example:
            >>> result = frame.group_by('flag', 'status').agg(
            ...     sum_qty=agg_sum('qty')
            ... )
        """
        from mxframe.groupby import MXGroupBy
        return MXGroupBy(self, keys, device=device)
    
    # ==================== Display ====================
    
    def __repr__(self) -> str:
        schema_str = ", ".join(str(c) for c in self.schema)
        return f"MXFrame({self.num_rows} rows, {self.num_columns} cols) [{schema_str}]"
    
    def __str__(self) -> str:
        return self.__repr__()
    
    def head(self, n: int = 5) -> 'MXFrame':
        """Return first n rows."""
        return MXFrame(self._table.slice(0, n))
    
    def tail(self, n: int = 5) -> 'MXFrame':
        """Return last n rows."""
        start = max(0, self.num_rows - n)
        return MXFrame(self._table.slice(start, n))

In [13]:
# Test MXFrame
table = pa.table({
    'qty': [1, 2, 3, 4, 5],
    'price': [10.5, 20.0, 30.5, 40.0, 50.5],
    'flag': [True, False, True, True, False],
})

frame = MXFrame(table)

print("=== MXFrame v0.0.1 ===")
print(f"Shape: {frame.shape}")
print(f"Rows: {frame.num_rows}")
print(f"Columns: {frame.column_names}")
print()
print("Schema:")
for col in frame.schema:
    print(f"  {col}")
print()
print(f"frame['qty'] = {frame['qty'].to_pylist()}")
print(f"head(2): {frame.head(2)}")

=== MXFrame v0.0.1 ===
Shape: (5, 3)
Rows: 5
Columns: ['qty', 'price', 'flag']

Schema:
  qty: int64?
  price: float64?
  flag: bool?

frame['qty'] = [1, 2, 3, 4, 5]
head(2): MXFrame(2 rows, 3 cols) [qty: int64?, price: float64?, flag: bool?]


In [14]:
#| hide
import nbdev; nbdev.nbdev_export()